In [ ]:
import networkx as nx
import pandas as pd


edges = pd.read_csv(r"C:\Users\Distinct HealthCare\Desktop\marvel-unimodal-edges.csv")
nodes = pd.read_csv(r"C:\Users\Distinct HealthCare\Desktop\带时代标签的节点表.csv")



# 后续计算代码不变
G = nx.from_pandas_edgelist(edges, source="Source", target="Target", create_using=nx.Graph())
degree = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)
eigenvector = nx.eigenvector_centrality_numpy(G)
clustering = nx.clustering(G)

nodes["Degree Centrality"] = nodes["Id"].map(degree)
nodes["Betweenness Centrality"] = nodes["Id"].map(betweenness)
nodes["Closeness Centrality"] = nodes["Id"].map(closeness)
nodes["Eigenvector Centrality"] = nodes["Id"].map(eigenvector)
nodes["Clustering Coefficient"] = nodes["Id"].map(clustering)

nodes.to_csv("marvel_node_centrality.csv", index=False)
import networkx as nx
import pandas as pd
import os

# ===================== 1. 读取文件（加路径验证）=====================
edges_path = r"C:\Users\Distinct HealthCare\Desktop\marvel-unimodal-edges.csv"
nodes_path = r"C:\Users\Distinct HealthCare\Desktop\带时代标签的节点表.csv"

# 验证文件是否存在
if not os.path.exists(edges_path):
    print(f" 边表不存在：{edges_path}")
else:
    print(" 成功读取边表")
    edges = pd.read_csv(edges_path)

if not os.path.exists(nodes_path):
    print(f" 节点表不存在：{nodes_path}")
else:
    print(" 成功读取节点表")
    nodes = pd.read_csv(nodes_path)

# ===================== 2. 构建网络（加进度提示）=====================
print("\n正在构建无向网络...")
G = nx.from_pandas_edgelist(edges, source="Source", target="Target", create_using=nx.Graph())
print(f 网络构建完成：节点数={G.number_of_nodes()}，边数={G.number_of_edges()}")

# ===================== 3. 计算5项中心性（加进度提示）=====================
print("\n正在计算度中心性...")
degree = nx.degree_centrality(G)

print("正在计算介数中心性（9000+边可能耗时1-3分钟）...")
betweenness = nx.betweenness_centrality(G)  # 最耗时的步骤

print("正在计算接近中心性...")
closeness = nx.closeness_centrality(G)

print("正在计算特征向量中心性...")
eigenvector = nx.eigenvector_centrality_numpy(G)

print("正在计算聚类系数...")
clustering = nx.clustering(G)

# ===================== 4. 合并数据（加验证）=====================
print("\n正在合并中心性指标到节点表...")
nodes["Degree Centrality"] = nodes["Id"].map(degree)
nodes["Betweenness Centrality"] = nodes["Id"].map(betweenness)
nodes["Closeness Centrality"] = nodes["Id"].map(closeness)
nodes["Eigenvector Centrality"] = nodes["Id"].map(eigenvector)
nodes["Clustering Coefficient"] = nodes["Id"].map(clustering)

# 验证是否成功添加列
print(f"指标合并完成！节点表列名：{list(nodes.columns)}")

# ===================== 5. 强制导出到桌面（加提示）=====================
output_path = r"C:\Users\Distinct HealthCare\Desktop\marvel_node_centrality.csv"
nodes.to_csv(output_path, index=False)
print(f"\n🎉 所有计算完成！结果文件已保存到：")
print(output_path)

# 验证文件是否生成
if os.path.exists(output_path):
    print(f"文件生成成功！文件大小：{os.path.getsize(output_path)/1024:.2f} KB")
else:
    print(f"文件生成失败！")


import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ----------------------
# 1. 读取你已经生成的中心性表格
# ----------------------
df = pd.read_csv("marvel_node_centrality.csv")

# ----------------------
# 2. 自动输出 5个中心性 TOP10
# ----------------------
metrics = [
    "Degree Centrality",
    "Betweenness Centrality",
    "Closeness Centrality",
    "Eigenvector Centrality",
    "Clustering Coefficient"
]

top10_dict = {}
for m in metrics:
    top10 = df.sort_values(by=m, ascending=False).head(10)
    top10_dict[m] = top10[["Label", "era", "gender", "team（复仇者联盟、银河护卫队、神盾局）", m]]
    top10_dict[m].to_csv(f"TOP10_{m}.csv", index=False, encoding="utf-8-sig")

print(" 已生成 5 个 TOP10 CSV 文件！")

# ----------------------
# 3. 按时代统计中心性（均值、中位数、最大）
# ----------------------
era_stats = df.groupby("era")[metrics].agg(["mean", "median", "max"]).round(4)
era_stats.to_csv("4个时代_中心性统计表.csv", encoding="utf-8-sig")
print(" 已生成 4个时代中心性统计表.csv！")

# ----------------------
# 4. 自动画图：5个指标的 TOP10 柱状图
# ----------------------
plt.rcParams["font.sans-serif"] = ["SimHei"]  # 显示中文
plt.rcParams["axes.unicode_minus"] = False

for m in metrics:
    top10 = df.sort_values(by=m, ascending=False).head(10)
    plt.figure(figsize=(12, 5))
    plt.barh(top10["Label"][::-1], top10[m][::-1], color="#1f77b4")
    plt.title(f"TOP10 | {m}", fontsize=14)
    plt.xlabel("中心性值")
    plt.tight_layout()
    plt.savefig(f"TOP10_{m}.png", dpi=300)
    plt.close()

print(" 已生成 5 张 TOP10 柱状图！")

# ----------------------
# 5. 自动画图：4个时代中心性均值对比图
# ----------------------
era_mean = df.groupby("era")[metrics].mean()

for m in metrics:
    plt.figure(figsize=(8, 4))
    era_mean[m].plot(kind="bar", color=["#ff9999","#66b3ff","#99ff99","#ffcc99"])
    plt.title(f"4个时代 {m} 均值对比")
    plt.ylabel("均值")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(f"时代对比_{m}.png", dpi=300)
    plt.close()

print(" 已生成 5 张 4时代对比图！")
print("\n 全部完成！你现在拥有：")
print("1. 5个TOP10表格（CSV）")
print("2. 4时代中心性统计表（CSV）")
print("3. 5张TOP10图 + 5张时代对比图（PNG）")

import networkx as nx
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

# ===================== 第一步：定义桌面路径（核心！）=====================
desktop_path = r"C:\Users\Distinct HealthCare\Desktop"
# 确保桌面路径存在
if not os.path.exists(desktop_path):
    desktop_path = os.path.expanduser("~/Desktop")  # 兜底

# ===================== 第二步：计算中心性并保存到桌面 =====================
# 1. 读取文件
edges_path = os.path.join(desktop_path, "marvel-unimodal-edges.csv")
nodes_path = os.path.join(desktop_path, "带时代标签的节点表.csv")

if not os.path.exists(edges_path):
    print(f" 边表不存在：{edges_path}")
else:
    print(" 成功读取边表")
    edges = pd.read_csv(edges_path)

if not os.path.exists(nodes_path):
    print(f"节点表不存在：{nodes_path}")
else:
    print("成功读取节点表")
    nodes = pd.read_csv(nodes_path)

# 2. 构建网络
print("\n 正在构建无向网络...")
G = nx.from_pandas_edgelist(edges, source="Source", target="Target", create_using=nx.Graph())
print(f"网络构建完成：节点数={G.number_of_nodes()}，边数={G.number_of_edges()}")

# 3. 计算5项中心性
print("\n正在计算度中心性...")
degree = nx.degree_centrality(G)
print("正在计算介数中心性（9000+边可能耗时1-3分钟）...")
betweenness = nx.betweenness_centrality(G)
print("正在计算接近中心性...")
closeness = nx.closeness_centrality(G)
print(" 正在计算特征向量中心性...")
eigenvector = nx.eigenvector_centrality_numpy(G)
print(" 正在计算聚类系数...")
clustering = nx.clustering(G)

# 4. 合并数据
print("\n 正在合并中心性指标到节点表...")
nodes["Degree Centrality"] = nodes["Id"].map(degree)
nodes["Betweenness Centrality"] = nodes["Id"].map(betweenness)
nodes["Closeness Centrality"] = nodes["Id"].map(closeness)
nodes["Eigenvector Centrality"] = nodes["Id"].map(eigenvector)
nodes["Clustering Coefficient"] = nodes["Id"].map(clustering)

# 5. 保存核心文件到桌面
core_file = os.path.join(desktop_path, "marvel_node_centrality.csv")
nodes.to_csv(core_file, index=False, encoding="utf-8-sig")
print(f"\n 核心文件已保存：{core_file}")

# ===================== 第三步：生成TOP10/时代统计/图表（全部保存到桌面）=====================
# 1. 读取核心文件
df = pd.read_csv(core_file)

# 2. 定义指标列表
metrics = [
    "Degree Centrality",
    "Betweenness Centrality",
    "Closeness Centrality",
    "Eigenvector Centrality",
    "Clustering Coefficient"
]

# 3. 生成TOP10表格（保存到桌面）
top10_dict = {}
for m in metrics:
    top10 = df.sort_values(by=m, ascending=False).head(10)
    # 修正列名：去掉括号（避免文件/列名报错）
    save_cols = ["Label", "era"]  # 保留核心列，避免列名不存在报错
    # 补充存在的列（防止你的节点表没有gender/team列）
    for col in ["gender", "team"]:
        if col in df.columns:
            save_cols.append(col)
    save_cols.append(m)
    
    top10_dict[m] = top10[save_cols]
    top10_file = os.path.join(desktop_path, f"TOP10_{m.replace(' ', '_')}.csv")
    top10_dict[m].to_csv(top10_file, index=False, encoding="utf-8-sig")
print(" 已生成 5 个 TOP10 CSV 文件（桌面）！")

# 4. 生成时代统计表（保存到桌面）
# 4. 生成时代统计表（保存到桌面）
era_stats = df_filtered.groupby("era")[metrics].agg(["mean", "median", "max"]).round(4)
era_stats_file = os.path.join(desktop_path, "4个时代_中心性统计表.csv")
era_stats.to_csv(era_stats_file, encoding="utf-8-sig")
print(" 已生成 4个时代中心性统计表.csv（桌面）！")

# 5. 生成TOP10柱状图（保存到桌面）
plt.rcParams["font.sans-serif"] = ["SimHei"]  # 显示中文
plt.rcParams["axes.unicode_minus"] = False

for m in metrics:
    top10 = df.sort_values(by=m, ascending=False).head(10)
    plt.figure(figsize=(12, 5))
    plt.barh(top10["Label"][::-1], top10[m][::-1], color="#1f77b4")
    plt.title(f"TOP10 | {m}", fontsize=14)
    plt.xlabel("中心性值")
    plt.tight_layout()
    img_file = os.path.join(desktop_path, f"TOP10_{m.replace(' ', '_')}.png")
    plt.savefig(img_file, dpi=300)
    plt.close()
print(" 已生成 5 张 TOP10 柱状图（桌面）！")

# 6. 生成时代对比图（保存到桌面）
# 先查看 era 有哪些值（可选，用于确认）
print("时代列的唯一值：", df["era"].unique())

# 过滤掉 Unknown Age（根据实际输出修改）
df_filtered = df[df["era"] != "Unknown Age"]
era_mean = df_filtered.groupby("era")[metrics].mean()

for m in metrics:
    plt.figure(figsize=(8, 4))
    era_mean[m].plot(kind="bar", color=["#ff9999","#66b3ff","#99ff99","#ffcc99"])
    plt.title(f"4个时代 {m} 均值对比", fontsize=12)
    plt.ylabel("均值")
    plt.xticks(rotation=30)
    plt.tight_layout()
    img_file = os.path.join(desktop_path, f"时代对比_{m.replace(' ', '_')}.png")
    plt.savefig(img_file, dpi=300)
    plt.close()

print(" 已生成 5 张 4时代对比图（桌面）！")
print("\n 全部完成！所有文件都在你的桌面：")
print(f" 桌面路径：{desktop_path}")
print("包含：")
print("1. marvel_node_centrality.csv（核心数据）")
print("2. 5个TOP10_XXX.csv（表格）")
print("3. 4个时代_中心性统计表.csv（表格）")
print("4. 10张PNG图片（TOP10图+时代对比图）")
    

✅ 成功读取边表
✅ 成功读取节点表

🔄 正在构建无向网络...
✅ 网络构建完成：节点数=327，边数=9891

🔄 正在计算度中心性...
🔄 正在计算介数中心性（9000+边可能耗时1-3分钟）...
🔄 正在计算接近中心性...
🔄 正在计算特征向量中心性...
🔄 正在计算聚类系数...

🔄 正在合并中心性指标到节点表...
✅ 指标合并完成！节点表列名：['Id', 'Label', 'gender', 'age', 'team（复仇者联盟、银河护卫队、神盾局）', 'team(正派 (Hero)、反派 (Villain)、中立 (Neutral) -主属性-只写英文）', 'race', 'first_appearance（剧中）', 'cleaned_year', 'era', 'Degree Centrality', 'Betweenness Centrality', 'Closeness Centrality', 'Eigenvector Centrality', 'Clustering Coefficient']

🎉 所有计算完成！结果文件已保存到：
C:\Users\Distinct HealthCare\Desktop\marvel_node_centrality.csv
✅ 文件生成成功！文件大小：68.64 KB
✅ 已生成 5 个 TOP10 CSV 文件！
✅ 已生成 4个时代中心性统计表.csv！
✅ 已生成 5 张 TOP10 柱状图！
✅ 已生成 5 张 4时代对比图！

🎉 全部完成！你现在拥有：
1. 5个TOP10表格（CSV）
2. 4时代中心性统计表（CSV）
3. 5张TOP10图 + 5张时代对比图（PNG）
✅ 成功读取边表
✅ 成功读取节点表

🔄 正在构建无向网络...
✅ 网络构建完成：节点数=327，边数=9891

🔄 正在计算度中心性...
🔄 正在计算介数中心性（9000+边可能耗时1-3分钟）...
🔄 正在计算接近中心性...
🔄 正在计算特征向量中心性...
🔄 正在计算聚类系数...

🔄 正在合并中心性指标到节点表...

✅ 核心文件已保存：C:\Users\Distinct HealthCare\Desktop\marvel_node_centrality.csv
✅ 已生成